# HW2 – Final pipeline 3. Final models

The last models for the ensemble, using the results of `01_data_tests` and `02_long_runs`:

1. **Train + val retraining** of the better long run, so that the final model uses all labeled data. It has no val set to be measured on, so only the public leaderboard can check it.
2. **Ensemble partner**, a model that makes different mistakes: a long tile RT-DETR if the tile test passed, otherwise YOLO11m.

Test predictions of all ensemble candidates are then saved, so that ensembles can be built later without a GPU.

## 0. Setup

In [ ]:
# Make the repo importable: Kaggle clones it, locally the root is two levels above notebooks/final_pipeline/
import os, sys
from pathlib import Path
if os.path.exists("/kaggle/working"):
    if not os.path.exists("/kaggle/working/repo"):
        !git clone --depth 1 https://github.com/BohdanZasymovych/ai-hw2-object-detection.git /kaggle/working/repo
    !pip install -q pycocotools huggingface_hub kaggle ultralytics==8.4.207   # only if missing; torch/torchvision stay Kaggle's own CUDA builds
    ROOT_DIR = Path("/kaggle/working/repo")
else:
    ROOT_DIR = Path.cwd().parents[1] if Path.cwd().parent.name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

In [ ]:
# Live reload of utils/common.py when editing locally (Kaggle's IPython autoreload is broken on Python 3.13)
if not os.path.exists("/kaggle/working"):
    %load_ext autoreload
    %autoreload 2
import json
import pandas as pd
import torch
import ultralytics
from utils.common import *
from utils.ultra import ULTRA_VERSION, YOLO_DATA_DIR, export_yolo, load_ultra, train_runs, ultra_predictor

seed_everything()
print(env_info(), "| ultralytics:", ultralytics.__version__, "| data:", RAW_DIR)
if ultralytics.__version__ != ULTRA_VERSION:
    print(f"Warning: tested with ultralytics {ULTRA_VERSION}; the patches in utils/ultra.py depend on its internals")

On Kaggle with **GPU T4 ×2** a Run All takes about 7.5 h. The training state is uploaded to the Hugging Face repo every 30 minutes, so running the notebook again after a lost session continues from there. Locally nothing is trained and the tables are rebuilt from the saved runs.

In [ ]:
QUICK = False                                    # True: local smoke test (few frames, 1 epoch, small input, nothing pushed)
ALL_STAGES = {"full", "partner", "test_preds"}
RUN = ALL_STAGES if os.path.exists("/kaggle/working") or QUICK else set()   # stages computed in this session; the rest are pulled from the HF repo
WINNER = None                                    # long run to retrain: None decides from the results of 02_long_runs, or "960" / "1280"
PARTNER = None                                   # ensemble partner: None decides from the results of 01_data_tests, or "tile" / "yolo"

# Fail early if trained stages could not be uploaded (needs HF_TOKEN with write access)
if RUN and not QUICK:
    check_hf_write()


def name(stage):
    return f"s3_{stage}" + ("_quick" if QUICK else "")


def earlier(run):
    # Name of a run from the earlier notebooks of the final pipeline
    return run + ("_quick" if QUICK else "")


def load_preds(run):
    return json.load(open(fetch_run(run) / "preds_val.json"))


def run_config(run):
    return json.load(open(fetch_run(run) / "config.json"))

## 1. Decisions from the earlier notebooks

**Long run:** the same rule as `02_long_runs`: the higher mAP; within 0.005, the higher mAP on val frames no wider than 1920 px, the sizes that occur in the test set.

**Partner:** the tile model if WBF of `rtdetr_final` and the short tile test beats WBF of `rtdetr_final` and YOLO11s, otherwise YOLO11m.

In [ ]:
val_ids = DetDataset("val", max_frames=8 if QUICK else None).image_ids
val_frames = subsample(load_split("val")[0])
test_like = val_frames[val_frames.image_id.isin(val_ids) & (val_frames.width <= 1920)].image_id.tolist()


def scores(preds):
    return coco_evaluate(preds, "val", image_ids=val_ids)["mAP"], coco_evaluate(preds, "val", image_ids=test_like)["mAP"]


long_preds = {s: load_preds(earlier(f"s2_{s}")) for s in ["960", "1280"]}
long_scores = {s: scores(p) for s, p in long_preds.items()}
tied = abs(long_scores["960"][0] - long_scores["1280"][0]) < 0.005
WINNER = WINNER or max(long_scores, key=lambda s: long_scores[s][1 if tied else 0])
OTHER = "960" if WINNER == "1280" else "1280"

if PARTNER is None:
    base = load_preds("rtdetr_final")
    with_tile = scores(wbf([base, load_preds(earlier("s1_tile"))]))[0]
    with_yolo = scores(wbf([base, load_preds("yolo_final")]))[0]
    PARTNER = "tile" if with_tile > with_yolo else "yolo"
    print(f"WBF with rtdetr_final: tile test {with_tile:.3f}, YOLO11s {with_yolo:.3f}")

print(f"long runs (mAP, test-like mAP): {long_scores}")
print(f"retraining: {WINNER} px | partner: {PARTNER}")

## 2. Training setup

**Train + val retraining:** the settings and data changes of the selected long run. There is no clean val set to stop on, so it trains for the same number of epochs as that run and keeps the last epoch. Val frames are frames of training here, so they get the same data changes.

**Partner:** a 6 h budget, like the long runs, without the data changes, because they were tested on full-frame RT-DETR only:
- **Tile RT-DETR:** the setup of the tile test in `01_data_tests`. Predictions come from 6 overlapping 960 px crops of each frame.
- **YOLO11m:** the setup of YOLO11s in notebook 04, except for batch 6 instead of 8, because the larger model needs more memory. Tested at 1280 and 1536 px.

In [ ]:
source = run_config(earlier(f"s2_{WINNER}"))
USED = set(source["used"])
LONG_SIDE = source["long_side"]
LOW_SIDES = (480, 320) if QUICK else (960, 680)     # low resolutions of the low-resolution simulation, as in 01_data_tests
FRAMES = 16 if QUICK else None                      # frames of each split (None: all)
BUDGET = dict(epochs=1, time=None) if QUICK else dict(epochs=100, time=6)
TILE, TILE_SIDE = (320, 640) if QUICK else (960, 1920)
COMMON = dict(optimizer="AdamW", scale=0.25, workers=2 if QUICK else 4, seed=SEED, max_det=EVAL_DETS, plots=False, cache=False)
RTDETR = dict(arch="rtdetr-l", weights="rtdetr-l.pt", **source["model"])
RTDETR_TRAIN = dict(batch=2 if QUICK else 4, lr0=1e-4, weight_decay=1e-4, nbs=16, warmup_epochs=0 if QUICK else 0.5, warmup_bias_lr=0.0,
                    close_mosaic=0 if QUICK else 2, deterministic=False, **COMMON)
PARTNERS = {
    "tile": dict(run=RTDETR, train=dict(imgsz=TILE, **RTDETR_TRAIN),
                 grid=[dict(imgsz=TILE, long_side=TILE_SIDE, nms=0.6, tile=TILE)]),
    "yolo": dict(run=dict(arch="yolo11m", weights="yolo11m.pt", p2=False),
                 train=dict(imgsz=320 if QUICK else 1280, batch=2 if QUICK else 6, lr0=0.001, warmup_epochs=0 if QUICK else 1,
                            close_mosaic=0 if QUICK else 2, deterministic=True, **COMMON),
                 grid=[dict(imgsz=s, long_side=s, nms=0.6) for s in ((320, 480) if QUICK else (1280, 1536))]),
}

## 3. Training

In [ ]:
def full_run():
    # Train + val retraining of the selected long run: same settings and number of epochs, last epoch kept
    changes = sorted(USED & {"ignore", "lowres"})
    folder = "_".join([*(["quick"] if QUICK else [str(LONG_SIDE)]), *(["full", *changes] if changes else [])])
    data = export_yolo(YOLO_DATA_DIR / folder, long_side=LONG_SIDE, max_frames=FRAMES, trainval=True, mask_ignore="ignore" in USED,
                       lowres={s: 0.1 for s in LOW_SIDES} if "lowres" in USED else None)
    epochs = len(json.load(open(fetch_run(earlier(f"s2_{WINNER}")) / "history.json")))
    train = {**RTDETR_TRAIN, "imgsz": source["input"], "epochs": epochs, "time": None, "val": False,
             **(dict(mosaic=0.5) if "mosaic" in USED else {})}
    return dict(name=name("full"), data=str(data), use_last=True, **RTDETR, train=train)


def partner_run():
    # Long run of the ensemble partner on data without the tested changes
    p = PARTNERS[PARTNER]
    if PARTNER == "tile":
        data = export_yolo(YOLO_DATA_DIR / ("quick_tile" if QUICK else "tile"), long_side=TILE_SIDE, max_frames=FRAMES, tile=TILE)
    else:
        data = export_yolo(YOLO_DATA_DIR / ("quick" if QUICK else str(LONG_SIDE)), long_side=LONG_SIDE, max_frames=FRAMES)
    return dict(name=name(PARTNER), data=str(data), **p["run"], train={**p["train"], **BUDGET})


def predict(model, split, imgsz, long_side, nms=0.7, tile=False):
    # Predictions of a model on a split (a few frames in the smoke test). Val keeps the usual score threshold;
    # test predictions are only used for WBF, which ignores scores below 0.01, and this keeps the files small
    ds = DetDataset(split, label_offset=0, long_side=long_side, max_frames=8 if QUICK and split == "val" else None)
    if QUICK and split == "test":
        ds.rows, ds.image_ids = ds.rows[:8], ds.image_ids[:8]
    preds, ms = predict_dataset(ultra_predictor(model, imgsz, nms=nms, tile=tile), ds, batch_size=1, num_workers=4,
                                score_thr=0.01 if split == "test" else 0.001)
    return preds, ds.image_ids, ms


def evaluate_partner():
    # The best test setting of the partner by val mAP; its predictions are saved
    model = load_ultra(run_dir(name(PARTNER)) / "best.pt")
    best = None
    for test in PARTNERS[PARTNER]["grid"]:
        preds, ids, ms = predict(model, "val", **test)
        m = {**coco_evaluate(preds, "val", image_ids=ids), "ms_per_img": ms}
        if best is None or m["mAP"] > best[1]["mAP"]:
            best = (test, m, preds, ids)
    save_run(name(PARTNER), config=dict(train=PARTNERS[PARTNER]["train"], test=best[0], image_ids=best[3]), metrics=best[1], preds=best[2])
    del model
    torch.cuda.empty_cache()


cfgs = ([full_run()] if "full" in RUN else []) + ([partner_run()] if "partner" in RUN else [])
if cfgs:
    train_runs(cfgs, checkpoint_min=None if QUICK else 30)
if "full" in RUN:
    save_run(name("full"), config=dict(source=f"s2_{WINNER}", used=sorted(USED), test=dict(imgsz=source["test"]["imgsz"], long_side=max(LONG_SIDE, source["test"]["imgsz"]))),
             metrics=dict(epochs=cfgs[0]["train"]["epochs"]))      # no val metrics: val frames were used for training
if "partner" in RUN:
    evaluate_partner()
if not QUICK:
    for stage, run in [("full", "full"), ("partner", PARTNER)]:
        if stage in RUN:
            push_run(name(run))

## 4. Comparison

The partner on the same val frames and evaluation as before, alone and in WBF with the selected long run. The train + val model cannot be evaluated here, because val is part of its training data.

In [ ]:
winner_preds, partner_preds = long_preds[WINNER], load_preds(name(PARTNER))
yolo_preds = load_preds("yolo_final")


def row(preds):
    m = coco_evaluate(preds, "val", image_ids=val_ids)
    return {"mAP": m["mAP"], "AP50": m["AP50"], "AP_small": m["AP_small"], "AR": m["AR"],
            "test-like mAP": coco_evaluate(preds, "val", image_ids=test_like)["mAP"]}


pd.DataFrame({
    f"long run {WINNER} px": row(winner_preds),
    f"partner: {PARTNER}": row(partner_preds),
    "WBF: long run + YOLO11s": row(wbf([winner_preds, yolo_preds])),
    f"WBF: long run + {PARTNER}": row(wbf([winner_preds, partner_preds])),
    f"WBF: both long runs + {PARTNER}": row(wbf([winner_preds, long_preds[OTHER], partner_preds])),
}).T.round(3)

## 5. Test predictions

Test predictions of every ensemble candidate, saved as `preds_test.json` in each run's folder and uploaded, so that the ensembles and the final submissions need no GPU.

In [ ]:
if "test_preds" in RUN:
    for run in [earlier(f"s2_{WINNER}"), earlier(f"s2_{OTHER}"), name(PARTNER), name("full")]:
        path = run_dir(run) / "preds_test.json"
        if path.exists():
            continue
        test = run_config(run)["test"]
        test = {"long_side": max(LONG_SIDE, test["imgsz"]), **test}
        model = load_ultra(fetch_run(run, weights=True) / "best.pt")
        preds, _, ms = predict(model, "test", **test)
        json.dump(preds, open(path, "w"))
        print(f"{run}: {len(preds)} test boxes, {ms:.0f} ms/img")
        if not QUICK:
            push_file(path, path_in_repo=f"runs/{run}/preds_test.json")
        del model
        torch.cuda.empty_cache()

## 6. Conclusion

To be written after the runs.